In [25]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
import joblib
import warnings
import time
warnings.filterwarnings("ignore")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import RootMeanSquaredError

In [26]:
df_train_production = pd.read_csv('df_train_production.csv')

In [27]:
df_train_production.drop(columns = ['row_id'], inplace = True)
df_train_production.shape

(1004424, 131)

In [28]:
df_train_production.dropna(inplace=True)

In [29]:
df_train_production['unit_target'] = df_train_production['target']/df_train_production['installed_capacity']
df_train_production['unit_target_48h'] = df_train_production['target_48h']/df_train_production['installed_capacity_48h']

In [30]:
df_train_production.dropna(inplace=True)

In [31]:
df_train_production.drop(columns=['year', 'is_consumption', 'day', 'installed_capacity', 'prediction_unit_id', 'target', 'eic_count'], inplace=True)

In [32]:
X_prod = df_train_production.drop(columns=['unit_target', 'datetime'])
y_prod = df_train_production[['unit_target', 'datetime']]

split_index = int(0.6 * len(X_prod))

X_train_prod, X_test_prod = X_prod[:split_index], X_prod[split_index:]
y_train_prod, y_test_prod = y_prod[:split_index], y_prod[split_index:]

In [33]:
top_features = ['surface_solar_radiation_downwards', 'surface_solar_radiation_downwards_min', 'is_business', 'unit_target_48h', 'total_precipitation_max', 'month', 'direct_solar_radiation_max', 'sin(hour)', 'product_type', 'total_precipitation', 'hour', 'is_country_holiday', 'cos(dayofyear)', 'total_precipitation_min', 'county', 'cloudcover_total_historical_grouped_by_date_48h', 'cloudcover_total_historical_mean_48h', 'target_168h', 'target_336h', 'cloudcover_low_max', 'cos(hour)', 'cloudcover_low_historical_grouped_by_date_48h', 'direct_solar_radiation', 'cloudcover_high_historical_mean_24h', 'installed_capacity_48h']
#Selecting the top N features from the dataset
X_train_prod_reduced = X_train_prod[top_features]
X_test_prod_reduced = X_test_prod[top_features]

In [34]:
len(top_features)

25

LSTM

In [35]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_prod_reduced)
X_test_scaled = scaler.transform(X_test_prod_reduced)

In [36]:
print('Training LSTM:')
X_train_lstm = np.reshape(X_train_scaled, (X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_lstm = np.reshape(X_test_scaled, (X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

optimizer = Adam(learning_rate=0.001)
early_stop = EarlyStopping(monitor="val_loss", mode='min', patience=10, restore_best_weights=True, verbose=2)

lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
lstm_model.add(Dense(1, activation='linear'))  

lstm_model.compile(optimizer=optimizer, loss='mse', metrics=[RootMeanSquaredError()])

start_time_tr = time.time()
lstm_model.fit(X_train_lstm, y_train_prod['unit_target'], epochs=100, batch_size=32, validation_split=0.2, verbose=2, callbacks=[early_stop])  
end_time_tr = time.time()

start_time_pr = time.time()
lstm_pred = lstm_model.predict(X_test_lstm)
end_time_pr = time.time()

training_time = end_time_tr - start_time_tr
prediction_time = end_time_pr - start_time_pr

mae = mean_absolute_error(y_test_prod['unit_target'], lstm_pred.flatten())
r2 = r2_score(y_test_prod['unit_target'], lstm_pred.flatten())
rmse = root_mean_squared_error(y_test_prod['unit_target'], lstm_pred.flatten())
mse = mean_squared_error(y_test_prod['unit_target'], lstm_pred.flatten())

results = {
    'Model': ['LSTM'],
    'Training Time (s)': [training_time],
    'Prediction Time (s)': [prediction_time],
    'MAE': [mae],
    'R-squared': [r2],
    'RMSE': [rmse],
    'MSE': [mse]
}

results_df = pd.DataFrame(results)

results_df.to_csv('lstm_metrics.csv', index=False)

joblib.dump(lstm_model, 'lstm_model.joblib')

print("Metrics saved to 'lstm_metrics.csv'")
print("Model saved as 'lstm_model.joblib'")
print(f"Training time: {training_time:.4f} seconds")
print(f"Prediction time: {prediction_time:.4f} seconds")

Training LSTM:
Epoch 1/100
14324/14324 - 11s - 756us/step - loss: 0.0024 - root_mean_squared_error: 0.0493 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0570
Epoch 2/100
14324/14324 - 10s - 690us/step - loss: 0.0021 - root_mean_squared_error: 0.0460 - val_loss: 0.0036 - val_root_mean_squared_error: 0.0604
Epoch 3/100
14324/14324 - 10s - 691us/step - loss: 0.0020 - root_mean_squared_error: 0.0448 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0569
Epoch 4/100
14324/14324 - 10s - 688us/step - loss: 0.0019 - root_mean_squared_error: 0.0439 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0569
Epoch 5/100
14324/14324 - 10s - 685us/step - loss: 0.0019 - root_mean_squared_error: 0.0432 - val_loss: 0.0034 - val_root_mean_squared_error: 0.0581
Epoch 6/100
14324/14324 - 10s - 693us/step - loss: 0.0018 - root_mean_squared_error: 0.0426 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0565
Epoch 7/100
14324/14324 - 10s - 686us/step - loss: 0.0018 - root_mean_squared_error: 0.0421